In [28]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

EPS = 1e-6

DATA_PATH = "../datasets/united.csv"
OUTPUT_PATH = "telemetry_with_energy_quantiles.csv"
CONFIG_PATH = "energy_segmentation_config.json"

SURFACE_HTML_DIR = Path("surface_data")
SURFACE_HTML_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [29]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

numeric_cols = [
    "depth",
    "d_depth",
    "d_time",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

print("Loaded:", df.shape)
print("Wells:", df["well_id"].nunique())
display(df.head())


Loaded: (415049, 9)
Wells: 1706


,processing_time,depth,d_depth,d_time,rotation,pressure_axis,pressure_rotation,well_id,speed
0,2025-08-24 10:09:50.980,0.0606,0.0303,11.0,74.256,804,4551,19601,0.002755
1,2025-08-24 10:10:00.260,0.0909,0.0303,10.0,73.812,763,3782,19601,0.003030
2,2025-08-24 10:10:05.199,0.1212,0.0303,5.0,73.512,879,4407,19601,0.006060
3,2025-08-24 10:10:14.610,0.1515,0.0303,11.0,73.962,721,3705,19601,0.002755
4,2025-08-24 10:10:34.197,0.1818,0.0303,20.0,74.256,859,3883,19601,0.001515


In [30]:
df["dt"] = df["d_time"]

df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]
df["pressure_balance"] = df["pressure_axis"] / (df["total_pressure"] + EPS)
df["axis_over_rot_pressure"] = df["pressure_axis"] / (df["pressure_rotation"] + EPS)
df["rot_pressure_over_axis"] = df["pressure_rotation"] / (df["pressure_axis"] + EPS)
df["rotation_efficiency"] = df["rotation"] / (df["pressure_rotation"] + EPS)

df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]

df["energy_input_proxy"] = df["pressure_axis"] + df["pressure_rotation"] * df["rotation"]
df["pseudo_mse"] = df["energy_input_proxy"] / (df["speed"] + EPS)

df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"])

display(
    df[
        [
            "speed",
            "pressure_axis",
            "pressure_rotation",
            "rotation",
            "energy_input_proxy",
            "pseudo_mse",
            "log_pseudo_mse",
        ]
    ].describe(percentiles=[.01, .05, .5, .95, .99])
)


,speed,pressure_axis,pressure_rotation,rotation,energy_input_proxy,pseudo_mse,log_pseudo_mse
count,415049.000000,415049.000000,415049.000000,415049.000000,4.150490e+05,4.150490e+05,415049.000000
mean,0.013116,17473.667273,14134.146325,103.945315,1.494500e+06,1.467727e+08,18.649861
std,0.006615,4682.982816,3243.523440,13.370361,4.009753e+05,9.155477e+07,0.552515
min,0.001002,317.000000,784.000000,50.010000,4.174649e+04,3.009300e+06,14.917218
1%,0.002755,3713.000000,6271.000000,64.980000,4.697380e+05,3.257078e+07,17.298926
5%,0.005050,6645.000000,8246.000000,81.750000,7.635049e+05,5.221720e+07,17.770923
50%,0.012120,18861.000000,14637.000000,103.158000,1.542361e+06,1.239899e+08,18.635711
95%,0.024240,22343.000000,18758.600000,138.474000,2.119089e+06,3.036335e+08,19.531332
99%,0.030300,23626.000000,20881.000000,139.020000,2.462635e+06,4.641533e+08,19.955725
max,0.038957,24872.000000,26318.000000,139.578000,3.668551e+06,2.581497e+09,21.671635


In [31]:
ROLLING_WINDOWS = (12, 30)
ROLLING_COLS = ["speed", "pseudo_mse", "log_pseudo_mse"]

for col in ROLLING_COLS:
    grp = df.groupby("well_id")[col]
    for window in ROLLING_WINDOWS:
        min_periods = max(3, window // 3)

        df[f"{col}_roll_median_{window}"] = grp.transform(
            lambda s: s.rolling(window, min_periods=min_periods).median()
        )
        df[f"{col}_roll_mean_{window}"] = grp.transform(
            lambda s: s.rolling(window, min_periods=min_periods).mean()
        )
        df[f"{col}_roll_std_{window}"] = grp.transform(
            lambda s: s.rolling(window, min_periods=min_periods).std()
        )

print("Rolling columns added.")


Rolling columns added.


In [32]:
def zscore(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / (series.std() + EPS)


df["hardness_score_smooth"] = zscore(df["log_pseudo_mse_roll_median_30"])

print("hardness_score_smooth NaN count:", int(df["hardness_score_smooth"].isna().sum()))
print("hardness_score_smooth NaN pct:", float(df["hardness_score_smooth"].isna().mean() * 100))

display(
    df[
        [
            "log_pseudo_mse",
            "log_pseudo_mse_roll_median_30",
            "hardness_score_smooth",
        ]
    ].describe(percentiles=[.01, .05, .5, .95, .99])
)


hardness_score_smooth NaN count: 15354
hardness_score_smooth NaN pct: 3.699322248698347


,log_pseudo_mse,log_pseudo_mse_roll_median_30,hardness_score_smooth
count,415049.000000,399695.000000,3.996950e+05
mean,18.649861,18.613582,-3.140151e-15
std,0.552515,0.417546,9.999976e-01
min,14.917218,16.156040,-5.885674e+00
1%,17.298926,17.493151,-2.683368e+00
5%,17.770923,17.892544,-1.726845e+00
50%,18.635711,18.650505,8.842769e-02
95%,19.531332,19.292167,1.625172e+00
99%,19.955725,19.476526,2.066702e+00
max,21.671635,20.164245,3.713749e+00


In [33]:
energy_labels_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

SEGMENT_SIZE = 60

segment_df = df.dropna(
    subset=["hardness_score_smooth", "pseudo_mse_roll_median_30", "speed"]
).copy()

segment_df = segment_df.sort_values(["well_id", "processing_time"]).copy()
segment_df["_row_in_well"] = segment_df.groupby("well_id").cumcount()
segment_df["segment_id"] = (segment_df["_row_in_well"] // SEGMENT_SIZE).astype(int)

segments = (
    segment_df
    .groupby(["well_id", "segment_id"])
    .agg(
        segment_start=("processing_time", "min"),
        segment_end=("processing_time", "max"),
        rows=("speed", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_30", "median"),
        speed_segment=("speed", "median"),
    )
    .reset_index()
)

segments["energy_type_segment_quantile"] = pd.qcut(
    segments["hardness_segment"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

segment_df = segment_df.merge(
    segments[["well_id", "segment_id", "hardness_segment", "energy_type_segment_quantile"]],
    on=["well_id", "segment_id"],
    how="left",
)

display(
    segments
    .groupby("energy_type_segment_quantile")
    .agg(
        segments=("segment_id", "size"),
        rows=("rows", "sum"),
        hardness_segment=("hardness_segment", "median"),
        pseudo_mse_segment=("pseudo_mse_segment", "median"),
        speed_segment=("speed_segment", "median"),
    )
    .sort_values("hardness_segment")
)


,segments,rows,hardness_segment,pseudo_mse_segment,speed_segment
energy_type_segment_quantile,,,,,
soft_low_energy,1876,100504,-1.024404,7.912285e+07,0.018180
medium_low_energy,1876,100086,-0.212240,1.110417e+08,0.012966
medium_high_energy,1875,98880,0.297587,1.373776e+08,0.012120
hard_high_energy,1876,100225,1.082099,1.908794e+08,0.006060


In [34]:
df_out = df.merge(
    segment_df[
        [
            "processing_time",
            "well_id",
            "segment_id",
            "hardness_segment",
            "energy_type_segment_quantile",
        ]
    ],
    on=["processing_time", "well_id"],
    how="left",
)

df_out["rock_energy_type_final"] = df_out["energy_type_segment_quantile"]

display(
    df_out
    .groupby("rock_energy_type_final")
    .agg(
        rows=("speed", "size"),
        speed_median=("speed", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_30", "median"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pressure_axis_median=("pressure_axis", "median"),
        pressure_rotation_median=("pressure_rotation", "median"),
        rotation_median=("rotation", "median"),
    )
    .sort_values("hardness_smooth")
)
display(df_out.head())


,rows,speed_median,pseudo_mse_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median
rock_energy_type_final,,,,,,,
soft_low_energy,100504,0.01818,7.826618e+07,-1.049528,16396.0,13275.0,103.410
medium_low_energy,100086,0.01212,1.108857e+08,-0.215161,19215.0,15170.0,103.008
medium_high_energy,98880,0.01212,1.373713e+08,0.298085,20396.0,15559.0,102.864
hard_high_energy,100225,0.00606,1.924607e+08,1.100641,19537.0,14351.0,103.458


,processing_time,depth,d_depth,d_time,rotation,pressure_axis,pressure_rotation,well_id,speed,dt,total_pressure,pressure_balance,axis_over_rot_pressure,rot_pressure_over_axis,rotation_efficiency,axis_x_rotation,rot_pressure_x_rotation,energy_input_proxy,pseudo_mse,log_energy_input_proxy,log_pseudo_mse,speed_roll_median_12,speed_roll_mean_12,speed_roll_std_12,speed_roll_median_30,speed_roll_mean_30,speed_roll_std_30,pseudo_mse_roll_median_12,pseudo_mse_roll_mean_12,pseudo_mse_roll_std_12,pseudo_mse_roll_median_30,pseudo_mse_roll_mean_30,pseudo_mse_roll_std_30,log_pseudo_mse_roll_median_12,log_pseudo_mse_roll_mean_12,log_pseudo_mse_roll_std_12,log_pseudo_mse_roll_median_30,log_pseudo_mse_roll_mean_30,log_pseudo_mse_roll_std_30,hardness_score_smooth,segment_id,hardness_segment,energy_type_segment_quantile,rock_energy_type_final
0,2025-08-24 10:09:50.980,0.0606,0.0303,11.0,74.256,804,4551,19601,0.002755,11.0,5355,0.150140,0.176664,5.660448,0.016316,59701.824,337939.056,338743.056,1.229314e+08,12.733000,18.627137,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,0.0909,0.0303,10.0,73.812,763,3782,19601,0.003030,10.0,4545,0.167877,0.201745,4.956750,0.019517,56318.556,279156.984,279919.984,9.235235e+07,12.542263,18.341122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-08-24 10:10:05.199,0.1212,0.0303,5.0,73.512,879,4407,19601,0.006060,5.0,5286,0.166288,0.199455,5.013652,0.016681,64617.048,323967.384,324846.384,5.359617e+07,12.691111,17.796988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-08-24 10:10:14.610,0.1515,0.0303,11.0,73.962,721,3705,19601,0.002755,11.0,4426,0.162901,0.194602,5.138696,0.019963,53326.602,274029.210,274750.210,9.970810e+07,12.523621,18.417758,0.002892,0.003650,0.001612,NaN,NaN,NaN,9.603023e+07,9.214701e+07,2.881584e+07,NaN,NaN,NaN,18.379440,18.295751,0.353801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-08-24 10:10:34.197,0.1818,0.0303,20.0,74.256,859,3883,19601,0.001515,20.0,4742,0.181147,0.221221,4.520373,0.019123,63785.904,288336.048,289195.048,1.907619e+08,12.574860,19.066537,0.002755,0.003223,0.001691,NaN,NaN,NaN,9.970810e+07,1.118700e+08,5.067291e+07,NaN,NaN,NaN,18.417758,18.449908,0.461198,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
SURFACE_GRID_SIZE = 40
SURFACE_MIN_BIN_COUNT = 5
SURFACE_SMOOTH_SIGMA = 1.6
SURFACE_Z_CLIP_QUANTILES = (0.02, 0.98)
SURFACE_AXIS_QUANTILES = (0.05, 0.95)

GLOBAL_SURFACE_PATH = SURFACE_HTML_DIR / "global_speed_surface.json"


def fill_and_smooth_surface(z: np.ndarray, counts: np.ndarray) -> tuple[np.ndarray, dict]:
    support_mask = counts >= SURFACE_MIN_BIN_COUNT
    z = z.astype(float, copy=True)
    z[~support_mask] = np.nan

    raw_filled_ratio = float(np.isfinite(z).mean())

    z_df = pd.DataFrame(z)
    z_df = z_df.interpolate(axis=0, limit_direction="both")
    z_df = z_df.interpolate(axis=1, limit_direction="both")

    z_filled = z_df.to_numpy(dtype=float)
    z_filled = np.where(np.isfinite(z_filled), z_filled, np.nanmedian(z_filled))

    weights = np.where(
        support_mask,
        np.clip(counts / max(SURFACE_MIN_BIN_COUNT, 1), 0.0, 1.0),
        0.0,
    )

    weighted_z = gaussian_filter(z_filled * weights, sigma=SURFACE_SMOOTH_SIGMA)
    weighted_w = gaussian_filter(weights, sigma=SURFACE_SMOOTH_SIGMA)

    z_smooth = np.where(weighted_w > EPS, weighted_z / weighted_w, z_filled)
    q_low, q_high = np.nanquantile(z_smooth, SURFACE_Z_CLIP_QUANTILES)
    z_smooth = np.clip(z_smooth, q_low, q_high)

    report = {
        "raw_filled_cell_ratio_after_min_count": raw_filled_ratio,
        "z_clip_low": float(q_low),
        "z_clip_high": float(q_high),
    }

    return z_smooth, report


def build_global_speed_surface(data: pd.DataFrame):
    x_col = "pressure_axis"
    y_col = "pressure_rotation"
    z_col = "speed"

    part = data[[x_col, y_col, z_col]].replace([np.inf, -np.inf], np.nan).dropna()
    part = part[part[z_col] > 0].copy()

    q_low, q_high = SURFACE_AXIS_QUANTILES
    x_min = float(part[x_col].quantile(q_low))
    x_max = float(part[x_col].quantile(q_high))
    y_min = float(part[y_col].quantile(q_low))
    y_max = float(part[y_col].quantile(q_high))

    part = part[part[x_col].between(x_min, x_max) & part[y_col].between(y_min, y_max)].copy()

    x_edges = np.linspace(x_min, x_max, SURFACE_GRID_SIZE + 1)
    y_edges = np.linspace(y_min, y_max, SURFACE_GRID_SIZE + 1)

    x_bin = np.clip(np.digitize(part[x_col], x_edges) - 1, 0, SURFACE_GRID_SIZE - 1)
    y_bin = np.clip(np.digitize(part[y_col], y_edges) - 1, 0, SURFACE_GRID_SIZE - 1)
    part["_x_bin"] = x_bin
    part["_y_bin"] = y_bin

    grouped = part.groupby(["_y_bin", "_x_bin"])[z_col].agg(["median", "count"]).reset_index()

    z = np.full((SURFACE_GRID_SIZE, SURFACE_GRID_SIZE), np.nan, dtype=float)
    counts = np.zeros((SURFACE_GRID_SIZE, SURFACE_GRID_SIZE), dtype=float)

    for y_idx, x_idx, z_median, z_count in grouped[["_y_bin", "_x_bin", "median", "count"]].to_numpy():
        z[int(y_idx), int(x_idx)] = float(z_median)
        counts[int(y_idx), int(x_idx)] = float(z_count)

    z_smooth, smooth_report = fill_and_smooth_surface(z, counts)

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    pa, pr = np.meshgrid(x_centers, y_centers)

    report = {
        "surface_name": "global_speed_surface",
        "x": x_col,
        "y": y_col,
        "z": z_col,
        "surface_method": "global_smoothed_empirical_binned_median_speed_surface",
        "rows_used_inside_axis_quantiles": int(len(part)),
        "grid_size": SURFACE_GRID_SIZE,
        "min_bin_count": SURFACE_MIN_BIN_COUNT,
        "smooth_sigma": SURFACE_SMOOTH_SIGMA,
        "axis_quantile_low": q_low,
        "axis_quantile_high": q_high,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "speed_min": float(np.nanmin(z_smooth)),
        "speed_median": float(np.nanmedian(z_smooth)),
        "speed_max": float(np.nanmax(z_smooth)),
        **smooth_report,
    }

    return pa, pr, z_smooth, report


global_pa, global_pr, global_z, global_surface_report = build_global_speed_surface(df_out)

GLOBAL_SURFACE_PATH.write_text(
    json.dumps(
        {
            "source": "global_speed_surface",
            "x": global_pa.tolist(),
            "y": global_pr.tolist(),
            "z": global_z.tolist(),
            "metadata": global_surface_report,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


print("Saved:", GLOBAL_SURFACE_PATH)


Saved: surface_data/global_speed_surface.json


/tmp/ipykernel_33361/3014111151.py:33: RuntimeWarning: invalid value encountered in divide
  z_smooth = np.where(weighted_w > EPS, weighted_z / weighted_w, z_filled)


In [36]:
front_cols = [
    "processing_time",
    "depth",
    "d_depth",
    "d_time",
    "rotation",
    "pressure_axis",
    "pressure_rotation",
    "well_id",
    "speed",
]
remaining_cols = [col for col in df_out.columns if col not in front_cols]
df_out = df_out[front_cols + remaining_cols]

df_out.to_csv(OUTPUT_PATH, index=False)

config = {
    "data_path": DATA_PATH,
    "output_path": OUTPUT_PATH,
    "final_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "segment_size": SEGMENT_SIZE,
    "labels": energy_labels_4,
    "hardness_features": ["hardness_score_smooth"],
    "hardness_base_signal": "log_pseudo_mse_roll_median_30",
    "hardness_formula": "zscore(log_pseudo_mse_roll_median_30)",
    "energy_quantile_source": "hardness_score_smooth segment median",
    "simulator_visual_surface_method": "global_smoothed_empirical_binned_median_speed_surface",
    "simulator_visual_surface_path": str(GLOBAL_SURFACE_PATH),
    "depth_column": "depth",
    "d_depth_column": "d_depth",
    "d_time_column": "d_time",
    "interpretation": {
        "hardness_score_smooth": "stable operational drilling resistance index based on smoothed log-pseudo-MSE",
        "rock_energy_type_final": "final discrete energy-response regime from segment-level quantile segmentation",
        "pseudo_mse": "proxy energy per penetration, not physical MSE",
        "simulator_visual_surface": "single global speed response surface; energy classes remain model features and context",
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Saved:", CONFIG_PATH)
print("Shape:", df_out.shape)
print("Columns head:", list(df_out.columns[:12]))


Saved: telemetry_with_energy_quantiles.csv
Saved: energy_segmentation_config.json
Shape: (415049, 44)
Columns head: ['processing_time', 'depth', 'd_depth', 'd_time', 'rotation', 'pressure_axis', 'pressure_rotation', 'well_id', 'speed', 'dt', 'total_pressure', 'pressure_balance']
